# Random Forest Model Training
This notebook contains the interactive equivalent of the `prediction_engine.py` logic. 
It demonstrates how to load consumer purchasing data, build a preprocessing pipeline for categorical variables, train a RandomForestRegressor from `scikit-learn`, and evaluate its predictions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib  # the standard way to export scikit-learn models

## 1. Load the Baseline Data
We load the baseline dataset generated by `data/generate_data.py`. We drop any rows that are missing the columns we need.

In [ ]:
baseline_df = pd.read_csv('data/baseline_data.csv')
df = baseline_df.dropna(subset=['Purchase_Amount', 'Product_Category', 'Payment_Method'])

print(f"Loaded {len(df)} rows from baseline data for training.")
display(df.head())

## 2. Setup the Pipeline
We use a `ColumnTransformer` to apply `OneHotEncoder` to the categorical inputs (`Product_Category`, `Payment_Method`). Then we chain this with the `RandomForestRegressor`.

In [ ]:
X = df[['Product_Category', 'Payment_Method']]
y = df['Purchase_Amount']

# 1. Create mapping process for categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Product_Category', 'Payment_Method'])
    ]
)

# 2. Build the full pipeline: encoding -> Random Forest
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42))
])

model

## 3. Train the Model

In [ ]:
print("Training model...")
model.fit(X, y)
print("Model training complete!")
print(f"Training R^2 Score: {model.score(X, y):.4f}")

## 4. Run a Sample Prediction
Let's predict the expected purchase amount for someone buying **Electronics** with a **Credit Card**.

In [ ]:
user_category = "Electronics"
user_payment = "Credit Card"

input_data = pd.DataFrame([{
    'Product_Category': user_category,
    'Payment_Method': user_payment
}])

prediction = model.predict(input_data)[0]

# Calculate standard deviation of predictions across all trees in the forest (for confidence intervals)
X_transformed = preprocessor.transform(input_data)
all_tree_preds = [tree.predict(X_transformed)[0] for tree in model.named_steps['regressor'].estimators_]
std_dev = np.std(all_tree_preds)

print(f"Sample Prediction ({user_category} + {user_payment}):")
print(f"Predicted Amount : ${prediction:.2f}")
print(f"Expected Range   : ${prediction - std_dev:.2f} to ${prediction + std_dev:.2f}")

## 5. View Feature Importances
Which inputs drove the random forest model the most?

In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
importances = model.named_steps['regressor'].feature_importances_

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False)

display(feat_df.head(10))

## 6. Export the Pre-Trained Model (Optional)
If you want to load this model manually in the dashboard without retraining it every time, you can dump it to a `.pkl` file.

In [ ]:
model_filename = 'random_forest_predictor.pkl'
joblib.dump(model, model_filename)
print(f"✅ Model successfully saved to {model_filename}")

# Note: to load it later, run:
# loaded_model = joblib.load('random_forest_predictor.pkl')